# Serial Frames for IRX4 Plus Multi Protocol Transmitter

## Flysky Protocol (AFHDS 1A)

### Simulate Channel 1

In [18]:
import time
import serial
import sys

def pack_channels_11bit(ch):
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_v1_frame_channels(
    sub_protocol,
    rx_num=0,
    type_value=0,
    option_value=0,
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    channels=None
):
    if channels is None:
        channels = [1024] * 16

    header = 0x55
    b1 = sub_protocol & 0x1F
    if bind: b1 |= 0x80
    if autobind: b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power: b2 |= 0x80

    b3 = option_value & 0xFF

    return bytes([header, b1, b2, b3]) + pack_channels_11bit(channels)

def print_frame_hex_once(frame, label="TX frame"):
    hexstr = " ".join(f"{b:02X}" for b in frame)
    print(f"{label} ({len(frame)} bytes):")
    print(hexstr)


# ------------------------
# Serial setup
# ------------------------

PORT = "/dev/ttyUSB0"
BAUD = 100000

PROTO_FLYSKY = 1
FLYSKY_SUBTYPE = 0

ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

print("Streaming… press Ctrl+C to stop")

channels = [1024] * 16
frame_hz = 50
period = 1.0 / frame_hz
t0 = time.time()


# Print exactly once
frame = make_multi_v1_frame_channels(
    sub_protocol=PROTO_FLYSKY,
    type_value=FLYSKY_SUBTYPE,
    channels=channels
)
print_frame_hex_once(frame)

try:
    while True:
        # simple CH1 sweep
        elapsed = time.time() - t0
        phase = (elapsed % 2.0) / 2.0
        lower = 500
        upper = 1500
        #channels[0] = int(204 + phase * (1843 - 204))
        channels[0] = int(lower + phase * (upper - lower))

        frame = make_multi_v1_frame_channels(
            sub_protocol=PROTO_FLYSKY,
            type_value=FLYSKY_SUBTYPE,
            channels=channels
        )

        ser.write(frame)
        time.sleep(period)

except KeyboardInterrupt:
    print("\nCtrl+C received — stopping transmitter")

finally:
    # optional: send neutral frames before exit
    neutral = [1024] * 16
    for _ in range(3):
        ser.write(make_multi_v1_frame_channels(
            PROTO_FLYSKY,
            type_value=FLYSKY_SUBTYPE,
            channels=neutral
        ))
        time.sleep(0.02)

    ser.close()
    print("Serial port closed cleanly")


Streaming… press Ctrl+C to stop
TX frame (26 bytes):
55 01 00 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80

Ctrl+C received — stopping transmitter
Serial port closed cleanly


In [9]:
import serial, time
ser = serial.Serial("/dev/ttyUSB0", 100000, bytesize=8, parity='E', stopbits=2)
try:
    while True:
        ser.write(b"\x55\xAA\x00\xFF")   # recognizable pattern
        time.sleep(0.01)
except KeyboardInterrupt:
    pass
finally:
    ser.close()


### Bind

In [17]:
# ------------------------
# Serial setup
# ------------------------

PORT = "/dev/ttyUSB0"
BAUD = 100000

PROTO_FLYSKY = 1
FLYSKY_SUBTYPE = 0

ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

print("Binding… press Ctrl+C to abort")

try:
    # ---- Bind phase ----
    start = time.time()
    while time.time() - start < 6.0:
        ser.write(make_multi_v1_frame_channels(
            sub_protocol=PROTO_FLYSKY,
            type_value=FLYSKY_SUBTYPE,
            bind=True
        ))
        time.sleep(0.02)

    print("Bind done → normal mode")

    # ---- Normal mode ----
    while True:
        ser.write(make_multi_v1_frame_channels(
            sub_protocol=PROTO_FLYSKY,
            type_value=FLYSKY_SUBTYPE
        ))
        time.sleep(0.02)

except KeyboardInterrupt:
    print("\nCtrl+C — exiting")

finally:
    ser.close()
    print("Serial closed")


Binding… press Ctrl+C to abort

Ctrl+C — exiting
Serial closed


## AFHDS2A Protocol

### Bind

In [31]:
import time
import serial

def pack_channels_11bit(ch):
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_v1_frame_channels(
    sub_protocol,
    rx_num=0,
    type_value=0,
    option_value=0,
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    channels=None
):
    if channels is None:
        channels = [1024] * 16

    header = 0x55
    b1 = sub_protocol & 0x1F
    if bind:       b1 |= 0x80
    if autobind:   b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    b3 = option_value & 0xFF
    return bytes([header, b1, b2, b3]) + pack_channels_11bit(channels)

def print_frame_hex_once(frame, label="TX frame"):
    print(f"{label} ({len(frame)} bytes):")
    print(" ".join(f"{b:02X}" for b in frame))

# Serial (MULTI serial mode: 100000 8E2)
PORT = "/dev/ttyUSB0"
BAUD = 100000
ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

PROTO_AFHDS2A = 28
AFHDS2A_PPM_IBUS = 1  # :contentReference[oaicite:8]{index=8}

# AFHDS2A binding note: use a different rx_num for each receiver you bind :contentReference[oaicite:9]{index=9}
rx_num = 0 #1 #0            # try 0..15 (and try another number if bind fails or if reusing an old bind)
option_value = 10      # 0=50Hz :contentReference[oaicite:10]{index=10}

frame_hz = 50
period = 1.0 / frame_hz
channels = [1024] * 16

print("AFHDS2A bind → run… Ctrl+C to stop")

try:
    # --- Binding phase ---
    bind_seconds = 6.0
    t0 = time.time()

    bind_frame = make_multi_v1_frame_channels(
        sub_protocol=PROTO_AFHDS2A,
        rx_num=rx_num,
        type_value=AFHDS2A_PPM_IBUS,
        option_value=option_value,
        bind=True,
        channels=channels
    )
    print_frame_hex_once(bind_frame, label="BIND frame (first one)")

    while time.time() - t0 < bind_seconds:
        ser.write(bind_frame)
        time.sleep(period)

    print("Bind phase complete. Switching to normal mode.")

    # --- Normal run phase (CH1 sweep) ---
    t1 = time.time()
    while True:
        elapsed = time.time() - t1
        phase = (elapsed % 2.0) / 2.0
        channels[0] = int(204 + phase * (1843 - 204))

        frame = make_multi_v1_frame_channels(
            sub_protocol=PROTO_AFHDS2A,
            rx_num=rx_num,
            type_value=AFHDS2A_PPM_IBUS,
            option_value=option_value,
            bind=False,
            channels=channels
        )
        ser.write(frame)
        time.sleep(period)

except KeyboardInterrupt:
    print("\nStopped (Ctrl+C).")

finally:
    ser.close()
    print("Serial closed.")


AFHDS2A bind → run… Ctrl+C to stop
BIND frame (first one) (26 bytes):
55 9C 10 0A 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80
Bind phase complete. Switching to normal mode.

Stopped (Ctrl+C).
Serial closed.


In [26]:
import time
import serial

def pack_channels_11bit(ch):
    """Pack 16x 11-bit channel values into 22 bytes (SBUS-style)."""
    if len(ch) != 16:
        raise ValueError("Need exactly 16 channels")
    out = bytearray(22)
    bitpos = 0
    for v in ch:
        if not (0 <= v <= 2047):
            raise ValueError("Channel values must be 0..2047")
        for b in range(11):
            if v & (1 << b):
                idx = (bitpos + b) // 8
                shift = (bitpos + b) % 8
                out[idx] |= (1 << shift)
        bitpos += 11
    return bytes(out)

def make_multi_v1_frame_channels(
    sub_protocol,          # 0..31 => header 0x55
    rx_num=0,              # important for AFHDS2A: different RXs must use different rx_num :contentReference[oaicite:3]{index=3}
    type_value=0,          # sub-protocol (0..7) placed in Stream[2] bits 4..6
    option_value=0,        # AFHDS2A: refresh rate option (0=50Hz, 70=400Hz) :contentReference[oaicite:4]{index=4}
    bind=False,
    autobind=False,
    rangecheck=False,
    low_power=False,
    channels=None
):
    if channels is None:
        channels = [1024] * 16

    header = 0x55
    b1 = sub_protocol & 0x1F
    if bind:       b1 |= 0x80
    if autobind:   b1 |= 0x40
    if rangecheck: b1 |= 0x20

    b2 = (rx_num & 0x0F) | ((type_value & 0x07) << 4)
    if low_power:
        b2 |= 0x80

    b3 = option_value & 0xFF  # signed int8 on wire, but ok to send as byte

    return bytes([header, b1, b2, b3]) + pack_channels_11bit(channels)

def print_frame_hex_once(frame, label="TX frame"):
    print(f"{label} ({len(frame)} bytes):")
    print(" ".join(f"{b:02X}" for b in frame))

# ------------------------
# Serial setup (MULTI = 100000 baud, 8E2)
# ------------------------
PORT = "/dev/ttyUSB0"   # change if needed
BAUD = 100000

ser = serial.Serial(
    PORT,
    BAUD,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_EVEN,
    stopbits=serial.STOPBITS_TWO,
    timeout=0
)

# ------------------------
# AFHDS2A settings
# ------------------------
PROTO_AFHDS2A = 28

# Sub-protocols (from MULTI docs):
# 0 PWM_IBUS
# 1 PPM_IBUS  <-- pick this for "PPM" variant
# 2 PWM_SBUS
# 3 PPM_SBUS
# 4 PWM_IBUS16
# 5 PPM_IBUS16
AFHDS2A_PPM_IBUS = 1  # :contentReference[oaicite:5]{index=5}

rx_num = 0            # try 0..15; also try different numbers if rebinding multiple RXs :contentReference[oaicite:6]{index=6}
option_value = 0      # 0=50Hz (safe start) :contentReference[oaicite:7]{index=7}

channels = [1024] * 16
frame_hz = 50
period = 1.0 / frame_hz
t0 = time.time()

print("AFHDS2A normal streaming… Ctrl+C to stop")

try:
    # Print one sample frame (once)
    channels[0] = 1024
    sample = make_multi_v1_frame_channels(
        sub_protocol=PROTO_AFHDS2A,
        rx_num=rx_num,
        type_value=AFHDS2A_PPM_IBUS,
        option_value=option_value,
        bind=False,
        channels=channels
    )
    print_frame_hex_once(sample, label="Sample AFHDS2A frame")

    # Stream continuously; modulate CH1 only
    while True:
        elapsed = time.time() - t0
        phase = (elapsed % 2.0) / 2.0     # 0..1 over 2 seconds
        channels[0] = int(204 + phase * (1843 - 204))  # ~-100%..+100%

        frame = make_multi_v1_frame_channels(
            sub_protocol=PROTO_AFHDS2A,
            rx_num=rx_num,
            type_value=AFHDS2A_PPM_IBUS,
            option_value=option_value,
            bind=False,
            channels=channels
        )
        ser.write(frame)
        time.sleep(period)

except KeyboardInterrupt:
    print("\nStopped (Ctrl+C).")

finally:
    # send a couple neutral frames then close
    neutral = [1024] * 16
    for _ in range(3):
        ser.write(make_multi_v1_frame_channels(
            PROTO_AFHDS2A, rx_num=rx_num, type_value=AFHDS2A_PPM_IBUS,
            option_value=option_value, bind=False, channels=neutral
        ))
        time.sleep(0.02)
    ser.close()
    print("Serial closed.")


AFHDS2A normal streaming… Ctrl+C to stop
Sample AFHDS2A frame (26 bytes):
55 1C 10 00 00 04 20 00 01 08 40 00 02 10 80 00 04 20 00 01 08 40 00 02 10 80

Stopped (Ctrl+C).
Serial closed.
